In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

d:\Code\Python\AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
class JokeState(TypedDict):
    joke: str
    topic: str
    explanation: str

In [3]:
model = ChatGroq(model='llama-3.3-70b-versatile')

In [4]:
def create_joke(state: JokeState) -> dict:
    joke = model.invoke(f"Tell me a joke about {state['topic']}")
    return {"joke": joke.content}

In [5]:
def explain_joke(state: JokeState) -> dict:
    explanation = model.invoke(f"Explain the joke: {state['joke']}")
    return {"explanation": explanation.content}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('create_joke', create_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'create_joke')
graph.add_edge('create_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpoint = InMemorySaver()                                #Adding persistence to the workflow by creating a checkpoint saver

joke_workflow = graph.compile(checkpointer=checkpoint)

### Create thread for workflow

In [7]:
config1 = {"configurable": {"thread_id": '1'}}

joke_workflow.invoke({'topic': 'police'}, config=config1)

{'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.',
 'topic': 'police',
 'explanation': 'A classic play on words. The joke is funny because it uses a common phrase "on the house" in a literal and figurative sense.\n\nIn this context, "on the house" is an idiom that means the drinks are free, usually courtesy of the host or the establishment. However, the phrase can also be interpreted literally, meaning something is physically located on the house, such as a roof or an upper floor.\n\nThe punchline is humorous because the police officer, having heard that the drinks are "on the house," brings a ladder to the party, implying that he thinks the drinks are literally located on the roof or an upper floor of the house, and he needs the ladder to access them. It\'s a clever and silly misunderstanding that creates the comedic effect.'}

### Get Final State of the workflow thread

In [8]:
joke_workflow.get_state(config1)

StateSnapshot(values={'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.', 'topic': 'police', 'explanation': 'A classic play on words. The joke is funny because it uses a common phrase "on the house" in a literal and figurative sense.\n\nIn this context, "on the house" is an idiom that means the drinks are free, usually courtesy of the host or the establishment. However, the phrase can also be interpreted literally, meaning something is physically located on the house, such as a roof or an upper floor.\n\nThe punchline is humorous because the police officer, having heard that the drinks are "on the house," brings a ladder to the party, implying that he thinks the drinks are literally located on the roof or an upper floor of the house, and he needs the ladder to access them. It\'s a clever and silly misunderstanding that creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 

### Get History of the workflow thread

In [9]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.', 'topic': 'police', 'explanation': 'A classic play on words. The joke is funny because it uses a common phrase "on the house" in a literal and figurative sense.\n\nIn this context, "on the house" is an idiom that means the drinks are free, usually courtesy of the host or the establishment. However, the phrase can also be interpreted literally, meaning something is physically located on the house, such as a roof or an upper floor.\n\nThe punchline is humorous because the police officer, having heard that the drinks are "on the house," brings a ladder to the party, implying that he thinks the drinks are literally located on the roof or an upper floor of the house, and he needs the ladder to access them. It\'s a clever and silly misunderstanding that creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '',

### Time Travel

In [ ]:
#Get intermediate state using checkpoint_id

joke_workflow.get_state({"configurable": {"thread_id": '1', "checkpoint_id": '1f15531a-685c-653a-8000-75785999edd6'}})

StateSnapshot(values={'topic': 'police'}, next=('create_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f15531a-685c-653a-8000-75785999edd6'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-21T16:25:16.073913+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15531a-6859-6b62-bfff-372115ae09b6'}}, tasks=(PregelTask(id='0f48f2b3-8514-b749-a2d5-2053b9eb4fe6', name='create_joke', path=('__pregel_pull', 'create_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.'}),), interrupts=())

### Re-Run For A Particular Checkpoint ID

In [11]:
joke_workflow.invoke(None,config={"configurable": {"thread_id": '1', "checkpoint_id": '1f15531a-685c-653a-8000-75785999edd6'}})

{'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.',
 'topic': 'police',
 'explanation': 'A classic play on words. The joke is funny because it sets up an expectation that the police officer brings a ladder to the party for a serious or practical reason, but the punchline subverts that expectation.\n\nThe phrase "the drinks are on the house" is a common idiomatic expression that means the drinks are free, usually courtesy of the host or the establishment. However, in this joke, the phrase is taken literally. The police officer hears that "the drinks are on the house," and he interprets it as the drinks being physically located on the house, perhaps on the roof or a high shelf. Therefore, he brings a ladder to access the drinks.\n\nThe humor comes from the unexpected twist on the common phrase and the silly idea that a police officer would bring a ladder to a party to get free drinks. It\'s a lighthearted and clever play o

In [12]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.', 'topic': 'police', 'explanation': 'A classic play on words. The joke is funny because it sets up an expectation that the police officer brings a ladder to the party for a serious or practical reason, but the punchline subverts that expectation.\n\nThe phrase "the drinks are on the house" is a common idiomatic expression that means the drinks are free, usually courtesy of the host or the establishment. However, in this joke, the phrase is taken literally. The police officer hears that "the drinks are on the house," and he interprets it as the drinks being physically located on the house, perhaps on the roof or a high shelf. Therefore, he brings a ladder to access the drinks.\n\nThe humor comes from the unexpected twist on the common phrase and the silly idea that a police officer would bring a ladder to a party to get free drinks. It\'s a lightheart

### Update State At Particular Checkpoint ID

In [15]:
joke_workflow.update_state({"configurable": {"thread_id": '1', "checkpoint_id": '1f15531a-685c-653a-8000-75785999edd6',  'checkpoint_ns': ''}}, {'topic': 'monkey'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15533a-13b0-6249-8001-bd928097052b'}}

In [16]:
list(joke_workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'monkey'}, next=('create_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15533a-13b0-6249-8001-bd928097052b'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-05-21T16:39:26.188807+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15531a-685c-653a-8000-75785999edd6'}}, tasks=(PregelTask(id='9f81693a-940f-a31f-978b-79df232ceeca', name='create_joke', path=('__pregel_pull', 'create_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'joke': 'Why did the police officer bring a ladder to the party?\n\nBecause he heard the drinks were on the house.', 'topic': 'police', 'explanation': 'A classic play on words. The joke is funny because it sets up an expectation that the police officer brings a ladder to the party for a serious or practical reason, but the punchline subverts that expectation.\n

### Re-Run Using The Updated State

In [17]:
joke_workflow.invoke(None,config={"configurable": {"thread_id": '1', "checkpoint_id": '1f15533a-13b0-6249-8001-bd928097052b'}})

{'joke': 'Why did the monkey get kicked out of the library?\n\nBecause he was caught monkeying around.',
 'topic': 'monkey',
 'explanation': 'A classic joke. The joke is funny because it\'s a play on words. "Monkeying around" is an idiomatic expression that means to fool around, cause trouble, or be mischievous. However, in this joke, it\'s a literal reference to a monkey (the animal) causing trouble in a library.\n\nThe punchline "he was caught monkeying around" is a clever use of wordplay, as it\'s a common phrase used to describe someone who\'s being unruly or disobedient, but in this case, it\'s applied to an actual monkey. The humor comes from the unexpected twist on the usual meaning of the phrase, creating a clever and silly connection between the setup and the punchline.'}